# Ensemble spatial-range time series

Opens a local NetCDF/Zarr ensemble or an Intake-ESM catalog entry, caches member spatial minima, means, and maxima, and plots their ensemble evolution.


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")

source_type = "member_netcdf"  # "member_netcdf", "netcdf", "zarr", or "intake_esm"
source = str(INPUT_DATA_ROOT / "CTRLEN10_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/EN*/archive/post/atm/180x360_aave/ts/daily/TREFHT.EN*.2012.nc")
storage_options = {}
dataset_key = None  # optional Intake-ESM dataset key
variable = "TREFHT"
member_dim = "member_id"
time_dim = "time"
time_slice = ("2012-01-01", "2012-02-29")  # None retains every input time
spatial_dims = None  # None infers every non-member, non-time dimension after level selection
level_dim = None  # inferred from lev, plev, or level when present
level = None  # None selects the last model level when the variable is 3-D
cache_tag = "CTRL10_S0_TREFHT_201201_201202"
force_compute = False
show = True


In [ ]:
import sys
import matplotlib.pyplot as plt
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))

from configs.output_paths import workflow_output_dirs
from util.ensemble_summary import (
    SUMMARY_STATISTICS,
    CACHE_FIELDS,
    open_ensemble_source,
    cache_ensemble_summary,
    plot_ensemble_summary,
)


## Notes: available variables and metrics

For `member_netcdf`, the source is a glob over files containing `EN##` in each filename; the notebook builds `member_dim` automatically. The default uses local TREFHT files and runs in the configured environment. Local NetCDF and optional Zarr inputs support any variable containing the configured member and time dimensions plus at least one spatial dimension. The optional Intake-ESM mode accepts any catalog variable but requires `intake-esm`; the legacy notebook demonstrated `PS`. Three-dimensional variables may select a positional level with an integer or a coordinate value with a float.

Cached member metrics are `spatial_min`, `spatial_mean`, and `spatial_max`. Their ensemble envelope and center fields are `ensemble_min`, `ensemble_mean`, and `ensemble_max`, each indexed by the spatial statistic. Existing cache files are reused without opening the raw or remote source unless `force_compute = True`.


In [ ]:
data_dir, figure_dir = workflow_output_dirs(
    "analysis_atm_da", "ensemble_ranges", output_root=DIAGNOSTIC_OUTPUT_ROOT
)
cache_path = data_dir / f"{cache_tag}_ensemble_summary.nc"
existed = cache_path.is_file()
dataset = None
if force_compute or not existed:
    dataset = open_ensemble_source(
        source,
        source_type=source_type,
        variable=variable,
        dataset_key=dataset_key,
        storage_options=storage_options,
        member_dim=member_dim,
    )
    if time_slice is not None:
        dataset = dataset.sel({time_dim: slice(*time_slice)})
summary = cache_ensemble_summary(
    cache_path,
    dataset=dataset,
    force_compute=force_compute,
    variable=variable,
    member_dim=member_dim,
    time_dim=time_dim,
    spatial_dims=spatial_dims,
    level_dim=level_dim,
    level=level,
)
if dataset is not None:
    dataset.close()
print("[REUSE]" if existed and not force_compute else "[WRITE]", cache_path)
fig = plot_ensemble_summary(summary, title=f"{variable} ensemble spatial ranges")
figure_path = figure_dir / f"{cache_tag}_ensemble_summary.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
if not show:
    plt.close(fig)
print("[FIGURE]", figure_path)
